# 代表坐标与客户卖家关联

本课沿用上一课的坐标候选规则，对每个「邮编前缀＋州」计算一对代表经纬度，再把代表点关联到客户和卖家原表。

| 输出 | 一行代表什么 | 主键 |
| --- | --- | --- |
| geo_reference | 一个邮编前缀在一个州中的代表坐标 | 邮编前缀＋州 |
| customers_with_geo | 一条客户记录及匹配结果 | customer_id |
| sellers_with_geo | 一条卖家记录及匹配结果 | seller_id |


## 1. 从原表读取

邮编和标识字段按字符串读取。客户和卖家的原表列用于比较关联前后的记录。


In [1]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display

project_dir = Path.cwd()
if project_dir.name == 'notebooks':
    project_dir = project_dir.parent
if not (project_dir / 'src').is_dir():
    raise ValueError('请从项目根目录或 notebooks 目录打开本课')

data_dir = project_dir.parent / 'archive'
read_options = {
    'dtype': str, 'encoding': 'utf-8-sig',
    'keep_default_na': False, 'na_values': [''],
}

geo_raw = pd.read_csv(data_dir / 'olist_geolocation_dataset.csv', **read_options)
customers_raw = pd.read_csv(data_dir / 'olist_customers_dataset.csv', **read_options)
sellers_raw = pd.read_csv(data_dir / 'olist_sellers_dataset.csv', **read_options)

state_reference = project_dir / 'docs' / 'references' / 'ibge_state_codes_v1.json'
state_codes = set(json.loads(state_reference.read_text(encoding='utf-8'))['state_codes'])
zip_col = 'geolocation_zip_code_prefix'
state_col = 'geolocation_state'
key_cols = [zip_col, state_col]


## 2. 重建候选坐标集

按上一课规则先去掉原字段整行重复，再筛选邮编、州和坐标。项目粗筛矩形沿用纬度 −35 到 6、经度 −75 到 −28。
保留城市标签供分组核查。 [粗筛参考的 IBGE 地理极点表](https://anuario.ibge.gov.br/images/aeb/2024/s1/2_pdf/s1t1101.pdf)


In [2]:
geo = geo_raw.drop_duplicates().copy()
geo[zip_col] = geo[zip_col].str.strip()
geo[state_col] = geo[state_col].str.strip().str.upper()
geo['city_label'] = geo['geolocation_city'].str.strip().str.lower().replace('', pd.NA)
geo['latitude'] = pd.to_numeric(geo['geolocation_lat'], errors='coerce')
geo['longitude'] = pd.to_numeric(geo['geolocation_lng'], errors='coerce')

# 缺失和解析失败会产生空值，between 对这些记录返回 False。
candidate_mask = (
    geo[zip_col].str.fullmatch('[0-9]{5}', na=False)
    & geo[state_col].isin(state_codes)
    & geo['latitude'].between(-35, 6)
    & geo['longitude'].between(-75, -28)
)
candidates = geo.loc[candidate_mask, key_cols + ['city_label', 'latitude', 'longitude']]
print('候选记录数', len(candidates))


候选记录数 738307


## 3. 每个邮编前缀和州生成一对代表坐标

每条候选记录在组内参与一次计算。分别取纬度和经度的中位数，得到邮编地区的近似代表点。
同时保留候选记录数和不同城市标签数，便于复查样本较少或标签较多的组。


In [3]:
geo_reference = candidates.groupby(key_cols, as_index=False).agg(
    representative_latitude=('latitude', 'median'),
    representative_longitude=('longitude', 'median'),
    candidate_record_count=('latitude', 'size'),
    candidate_city_label_count=('city_label', 'nunique'),
)

# 两列一起构成主键。相同邮编前缀在不同州仍可各有一行。
if geo_reference[key_cols].isna().any().any() or geo_reference.duplicated(key_cols).any():
    raise ValueError('代表坐标表的组合主键缺失或重复')
if geo_reference['candidate_record_count'].sum() != len(candidates):
    raise ValueError('各组候选记录数的合计与来源不一致')

print('代表坐标表行数', len(geo_reference))
display(geo_reference.loc[
    geo_reference[zip_col].isin(['02116', '28155'])
])


代表坐标表行数 19019


,geolocation_zip_code_prefix,geolocation_state,representative_latitude,representative_longitude,candidate_record_count,candidate_city_label_count
367,02116,RN,-23.515978,-46.582170,1,1
368,02116,SP,-23.519402,-46.584880,10,2
7605,28155,RJ,-22.245042,-43.695819,4,1


## 4. 用组合键做多对一左关联

同一个地区组合可以对应多条客户或卖家记录。右侧代表坐标表每组只有一行，many_to_one 会校验这项要求。
geo_match 的 both 表示匹配到候选坐标，left_only 表示只保留左表记录，新增坐标为空。


In [4]:
join_keys = ['geo_zip_key', 'geo_state_key']
lookup = geo_reference.rename(columns=dict(zip(key_cols, join_keys)))

def attach_geo(raw, entity):
    id_col = entity + '_id'
    source_zip = entity + '_zip_code_prefix'
    source_state = entity + '_state'

    if raw[id_col].isna().any() or raw[id_col].duplicated().any():
        raise ValueError('来源 ID 缺失或重复')

    # 统一临时键的空白和大小写，原始地区字段仍可用于核查。
    left = raw.assign(
        geo_zip_key=raw[source_zip].str.strip(),
        geo_state_key=raw[source_state].str.strip().str.upper(),
    )
    joined = left.merge(
        lookup, on=join_keys, how='left',
        validate='many_to_one', indicator='geo_match',
    ).drop(columns=join_keys)

    # 同时核对行数、顺序和所有来源字段。
    if not joined[raw.columns].equals(raw.reset_index(drop=True)):
        raise ValueError('关联后的来源记录与输入不一致')
    return joined

customers_with_geo = attach_geo(customers_raw, 'customer')
sellers_with_geo = attach_geo(sellers_raw, 'seller')


## 5. 核对行数和坐标缺失

用记录数核对关联前后的行数，再把输出拆成匹配成功和找不到候选坐标两组。
两组之和应等于原表记录数。


In [5]:
def geo_join_controls(raw, joined, id_col):
    matched = joined['geo_match'].eq('both')
    coordinate_cols = ['representative_latitude', 'representative_longitude']
    missing_coordinate = joined[coordinate_cols].isna().any(axis=1)
    if not missing_coordinate.equals(~matched):
        raise ValueError('坐标缺失状态与关联结果不一致')
    return pd.Series({
        '关联前行数': len(raw),
        '关联后行数': len(joined),
        '行数差额': len(joined) - len(raw),
        '关联后主键重复数': joined[id_col].duplicated().sum(),
        '匹配到代表坐标': matched.sum(),
        '未匹配到代表坐标': (~matched).sum(),
    })

join_controls = pd.DataFrame({
    '客户表': geo_join_controls(customers_raw, customers_with_geo, 'customer_id'),
    '卖家表': geo_join_controls(sellers_raw, sellers_with_geo, 'seller_id'),
})
display(join_controls)

# 检查几条未匹配的卖家记录。坐标为空时仍可以保留卖家 ID 和业务记录。
display(sellers_with_geo.loc[
    sellers_with_geo['geo_match'].eq('left_only'),
    ['seller_zip_code_prefix', 'seller_state', 'representative_latitude', 'representative_longitude']
].head())


,客户表,卖家表
关联前行数,99441,3095
关联后行数,99441,3095
行数差额,0,0
关联后主键重复数,0,0
匹配到代表坐标,99162,3053
未匹配到代表坐标,279,42


,seller_zip_code_prefix,seller_state,representative_latitude,representative_longitude
70,95076,SP,NaN,NaN
114,21210,RN,NaN,NaN
198,85301,SP,NaN,NaN
206,87360,SP,NaN,NaN
275,86170,SP,NaN,NaN


## 我的理解

逐格运行后，发来最后一格的关联核对表，并回答两题。

1. 代表坐标表的一行表示什么？为什么它能作为 many_to_one 的右表？
每一行表示一个邮编和州名代表坐标以及候选记录数、候选城市数，每个邮编前缀和州的组合只出现一次。
2. 客户和卖家都有未匹配记录，为什么关联后的总行数仍然等于原表行数？
左关联会保留每条客户和卖家记录。右表每组最多匹配一行，所以匹配到和匹配不到的记录都各占一行。
核对后会把地理阶段的规则、代表坐标和覆盖报告整理成可重复运行的流程。
